# 01 - Data Exploration
Exploring `data/processed/cleaned_dataset.csv` before feature engineering and modelling.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/processed/cleaned_dataset.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.shape

## 1. Basic overview

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print('Rows with imputed max_temp:', df['max_temp_imputed'].sum(), f"({df['max_temp_imputed'].mean()*100:.2f}%)")
print('Rows with imputed rainfall:', df['rainfall_imputed'].sum(), f"({df['rainfall_imputed'].mean()*100:.2f}%)")

## 2. Time patterns in CO2

In [ ]:
hourly = df.groupby('hour_of_day')['co2_estimate'].mean()
hourly.plot(kind='line', marker='o', figsize=(10,4), title='Mean CO2 estimate by hour of day')
plt.ylabel('kg CO2-e')
plt.xlabel('Hour')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily = df.groupby('day_of_week')['co2_estimate'].mean().reindex(day_order)
daily.plot(kind='bar', figsize=(8,4), title='Mean CO2 estimate by day of week')
plt.ylabel('kg CO2-e')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('is_weekend')['co2_estimate'].mean()

## 3. Comparison across stations

In [ ]:
df.groupby('road')['co2_estimate'].agg(['mean','median','std']).round(1)

In [ ]:
for road in df['road'].unique():
    subset = df[df['road']==road].set_index('timestamp')['co2_estimate']
    daily_avg = subset.resample('D').mean()
    daily_avg.plot(figsize=(12,4), label=road, alpha=0.7)
plt.legend()
plt.title('Daily average CO2 estimate over time, by station')
plt.ylabel('kg CO2-e')
plt.show()

## 4. Correlation with target

**Important:** `volume_light` and `volume_heavy` are expected to correlate very strongly with `co2_estimate` since they are direct inputs to the formula that computes it (see DATA.md, "Target variable" section). This is not a meaningful discovery for RQ1/RQ4 — it's circular by construction. The features actually worth interpreting via SHAP are the ones *not* used to compute the target: weather and temporal features.

In [ ]:
numeric_cols = ['volume_light','volume_heavy','max_temp','rainfall','hour_of_day','month','co2_estimate']
corr = df[numeric_cols].corr()
corr['co2_estimate'].sort_values(ascending=False)

In [ ]:
import numpy as np
fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_yticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha='right')
ax.set_yticklabels(numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()

## 5. Weather relationship (excluding volume)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].scatter(df['max_temp'], df['co2_estimate'], alpha=0.1, s=5)
axes[0].set_xlabel('Max temperature (C)')
axes[0].set_ylabel('CO2 estimate (kg)')
axes[0].set_title('CO2 vs max temperature')

axes[1].scatter(df['rainfall'], df['co2_estimate'], alpha=0.1, s=5)
axes[1].set_xlabel('Rainfall (mm)')
axes[1].set_ylabel('CO2 estimate (kg)')
axes[1].set_title('CO2 vs rainfall')
plt.tight_layout()
plt.show()